In [2]:

# ============================================================
# MISSION: AI ENGINEER 2027
# PROJECT: SMS SPAM DETECTION USING NLP
# TF-IDF + MULTINOMIAL NAIVE BAYES
# ============================================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# STEP 1: LOAD THE DATASET
url = (
    "https://raw.githubusercontent.com/"
    "justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
)

df = pd.read_csv(
    url,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("=" * 60)
print("1. DATASET LOADED")
print("=" * 60)
print(df.head())
print("\nDataset shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())


# STEP 2: CHECK AND CLEAN DATA
df = df.dropna(subset=["label", "message"]).copy()

df["label"] = df["label"].str.strip().str.lower()
df["message"] = df["message"].astype(str).str.strip()

df = df[
    df["label"].isin(["ham", "spam"]) &
    (df["message"] != "")
].copy()

df = df.drop_duplicates(subset=["label", "message"])

print("\n" + "=" * 60)
print("2. CLEANED DATASET")
print("=" * 60)
print("Rows after cleaning:", len(df))
print(df["label"].value_counts())


# STEP 3: SEPARATE FEATURES AND LABELS
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 60)
print("3. TRAIN-TEST SPLIT")
print("=" * 60)
print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))


# STEP 4: BUILD THE NLP + MACHINE LEARNING PIPELINE
model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB())
])


# STEP 5: TRAIN THE MODEL
model.fit(X_train, y_train)

print("\n" + "=" * 60)
print("4. MODEL TRAINED SUCCESSFULLY!")
print("=" * 60)


# STEP 6: PREDICT ON UNSEEN TEST DATA
y_pred = model.predict(X_test)

print("\n" + "=" * 60)
print("5. MODEL EVALUATION")
print("=" * 60)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=["ham", "spam"],
    zero_division=0
))

print("Confusion Matrix (rows = actual, columns = predicted)")
print("Label order: ham, spam")
print(confusion_matrix(y_test, y_pred, labels=["ham", "spam"]))


# STEP 7: TEST YOUR OWN MESSAGES
test_messages = [
    "Congratulations! You have won a free prize. Claim now!",
    "Hey bro, are we meeting at college tomorrow?",
    "URGENT! You have won a cash reward. Call now!",
    "Please send me the class notes when you are free."
]

predictions = model.predict(test_messages)

print("\n" + "=" * 60)
print("6. CUSTOM MESSAGE PREDICTIONS")
print("=" * 60)

for message, prediction in zip(test_messages, predictions):
    print(f"\nMessage: {message}")
    print(f"Prediction: {prediction.upper()}")


# STEP 8: TEST A SINGLE NEW MESSAGE
new_message = ["Free entry in a prize draw! Reply now!"]

prediction = model.predict(new_message)[0]
probabilities = model.predict_proba(new_message)[0]

classes = model.named_steps["classifier"].classes_
probability_map = dict(zip(classes, probabilities))

print("\n" + "=" * 60)
print("7. SINGLE MESSAGE TEST")
print("=" * 60)
print("Message:", new_message[0])
print("Predicted label:", prediction.upper())
print("Model estimated probabilities:")

for label, probability in probability_map.items():
    print(f"{label.upper()}: {probability:.2%}")

print("\nPROJECT RUN COMPLETED!")




import joblib
from pathlib import Path

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

joblib.dump(model, model_dir / "sms_spam_model.pkl")

print("SpamShield AI model saved!")

1. DATASET LOADED
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset shape: (5572, 2)

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

2. CLEANED DATASET
Rows after cleaning: 5158
label
ham     4516
spam     642
Name: count, dtype: int64

3. TRAIN-TEST SPLIT
Training messages: 4126
Testing messages: 1032

4. MODEL TRAINED SUCCESSFULLY!

5. MODEL EVALUATION
Accuracy: 0.9506

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       904
        spam       1.00      0.60      0.75       128

    accuracy                           0.95      1032
   macro avg       0.97      0.80      